In [692]:
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

## Configuration

In [693]:
RANDOM_STATE=42
ROOT_DIR = Path().cwd().parent
DATASET_DIR = ROOT_DIR / "dataset"
CSV_FILE_PATH = Path.joinpath(DATASET_DIR,"processed","classification_athlete_events.csv")
print(ROOT_DIR,"\n",DATASET_DIR,"\n",CSV_FILE_PATH)

/home/mdev/Documents/ml/olympic_medals_prediction 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset 
 /home/mdev/Documents/ml/olympic_medals_prediction/dataset/processed/classification_athlete_events.csv


## Load dataset and modify

In [694]:
df = pd.read_csv(CSV_FILE_PATH)
df.head()

,noc,season,year,athletes,athletes_female_percentage,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals,km_cluster,ag_cluster
0,CHN,1,1932,1,0.000000,22.0,175.0,70.0,0.0,0.0,1,2,0,0,0
1,CHN,1,1936,54,0.037037,24.0,175.0,69.0,0.0,0.0,7,27,0,0,0
2,CHN,1,1948,31,0.032258,26.0,175.0,70.0,0.0,0.0,6,12,0,0,0
3,CHN,1,1952,1,0.000000,23.0,175.0,70.0,0.0,0.0,1,1,0,0,0
4,CHN,1,1984,215,0.386047,23.0,173.0,66.0,0.0,0.0,19,105,32,0,0


In [695]:
df = df.drop(columns=['athletes_female_percentage','ag_cluster'])
df = df.rename(columns={'km_cluster':'label'})
df.head()

,noc,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals,label
0,CHN,1,1932,1,22.0,175.0,70.0,0.0,0.0,1,2,0,0
1,CHN,1,1936,54,24.0,175.0,69.0,0.0,0.0,7,27,0,0
2,CHN,1,1948,31,26.0,175.0,70.0,0.0,0.0,6,12,0,0
3,CHN,1,1952,1,23.0,175.0,70.0,0.0,0.0,1,1,0,0
4,CHN,1,1984,215,23.0,173.0,66.0,0.0,0.0,19,105,32,0


## Class ratio

In [696]:
class_ratio = df['label'].value_counts(normalize=True)
class_ratio

label
0    0.898358
1    0.089393
2    0.012249
Name: proportion, dtype: float64

In [697]:
df[df['label'] == 0]

,noc,season,year,athletes,avg_age,agv_height,avg_weight,prev_medals,prev_gold_medals,sports,events,total_medals,label
0,CHN,1,1932,1,22.0,175.0,70.0,0.0,0.0,1,2,0,0
1,CHN,1,1936,54,24.0,175.0,69.0,0.0,0.0,7,27,0,0
2,CHN,1,1948,31,26.0,175.0,70.0,0.0,0.0,6,12,0,0
3,CHN,1,1952,1,23.0,175.0,70.0,0.0,0.0,1,1,0,0
4,CHN,1,1984,215,23.0,173.0,66.0,0.0,0.0,19,105,32,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3832,MNE,0,2014,2,19.0,180.0,73.0,0.0,0.0,1,3,0,0
3833,ALB,0,2006,1,19.0,180.0,74.0,0.0,0.0,1,3,0,0
3834,ALB,0,2010,1,23.0,180.0,74.0,0.0,0.0,1,2,0,0
3835,ALB,0,2014,1,20.0,163.0,56.0,0.0,0.0,1,2,0,0


## Split dataset into train,test

In [698]:
X = df.drop(columns=['noc','label','prev_medals','prev_gold_medals','total_medals'])
y = df['label']
X.head()

,season,year,athletes,avg_age,agv_height,avg_weight,sports,events
0,1,1932,1,22.0,175.0,70.0,1,2
1,1,1936,54,24.0,175.0,69.0,7,27
2,1,1948,31,26.0,175.0,70.0,6,12
3,1,1952,1,23.0,175.0,70.0,1,1
4,1,1984,215,23.0,173.0,66.0,19,105


In [699]:
y

0       0
1       0
2       0
3       0
4       0
       ..
3832    0
3833    0
3834    0
3835    0
3836    0
Name: label, Length: 3837, dtype: int64

In [700]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=RANDOM_STATE,stratify=y)

In [701]:
train,test = train_test_split(X,test_size=0.2,random_state=RANDOM_STATE)

## Scaling for logistic regression only

In [702]:
scaler = RobustScaler()

In [703]:
print(train.head())
print(test.head())

      season  year  athletes  avg_age  agv_height  avg_weight  sports  events
433        1  1920        47     25.0       176.0        71.0       8      34
1569       1  1960        65     28.0       172.0        71.0      11      52
371        1  2000       435     26.0       178.0        72.0      30     238
3148       0  1984        99     25.0       175.0        72.0      10      38
2298       1  1912         2     19.0       172.0        65.0       1       2
      season  year  athletes  avg_age  agv_height  avg_weight  sports  events
1113       1  2008         5     24.0       177.0        70.0       3       5
1309       1  2008        21     25.0       179.0        73.0       5       7
805        1  1920        39     29.0       176.0        70.0       7      34
1173       1  1968       226     25.0       175.0        70.0      18     124
2770       1  1996         6     23.0       173.0        67.0       4       6


In [704]:
exclude_cols = ['season']
to_scale_cols = [ x for x in train.columns.tolist() if x not in exclude_cols ]
to_scale_cols

['year', 'athletes', 'avg_age', 'agv_height', 'avg_weight', 'sports', 'events']

In [705]:
train[to_scale_cols] = scaler.fit_transform(train[to_scale_cols])
test[to_scale_cols] = scaler.transform(test[to_scale_cols])

In [706]:
print(train.head())
print(test.head())

      season      year  athletes   avg_age  agv_height  avg_weight  sports  \
433        1 -2.000000  0.653061  0.000000        0.25    0.166667   0.375   
1569       1 -0.888889  1.020408  1.000000       -0.75    0.166667   0.750   
371        1  0.222222  8.571429  0.333333        0.75    0.333333   3.125   
3148       0 -0.222222  1.714286  0.000000        0.00    0.333333   0.625   
2298       1 -2.222222 -0.265306 -2.000000       -0.75   -0.833333  -0.500   

        events  
433   0.733333  
1569  1.333333  
371   7.533333  
3148  0.866667  
2298 -0.333333  
      season      year  athletes   avg_age  agv_height  avg_weight  sports  \
1113       1  0.444444 -0.204082 -0.333333        0.50         0.0  -0.250   
1309       1  0.444444  0.122449  0.000000        1.00         0.5   0.000   
805        1 -2.000000  0.489796  1.333333        0.25         0.0   0.250   
1173       1 -0.666667  4.306122  0.000000        0.00         0.0   1.625   
2770       1  0.111111 -0.183673 -0.666

In [707]:
t_train,t_test = train_test_split(y,test_size=0.2,random_state=RANDOM_STATE)

## Model training

In [708]:
lr = LogisticRegression(max_iter=1000,random_state=RANDOM_STATE,class_weight="balanced")
lr.fit(train,t_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [709]:
sample_weights = compute_sample_weight(class_weight="balanced",y=y_train)

In [710]:
xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.1,
    random_state=RANDOM_STATE
)
xgb.fit(X_train,y_train,sample_weight=sample_weights)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [711]:
rf = RandomForestClassifier(random_state=RANDOM_STATE,n_estimators=500,class_weight="balanced",max_depth=8)
rf.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 T

## Model Predictions

In [712]:
y_pred = xgb.predict(X_test)
y_pred[:15]

array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0])

In [713]:
rf_pred = rf.predict(X_test)
rf_pred[:15]

array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0])

In [714]:
lr_pred = lr.predict(test)
lr_pred[:15]

array([0, 0, 1, 1, 0, 0, 0, 1, 0, 2, 1, 0, 1, 0, 1])

In [715]:
def print_metrics(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n{name} Performance:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall   : {rec:.3f}")
    print(f"  F1-Score : {f1:.3f}")

In [716]:
print_metrics("XGboost",y_test, y_pred)
print_metrics("Random forest",y_test, rf_pred)
print_metrics("Logistic regression",t_test, lr_pred)


XGboost Performance:
  Accuracy : 0.947
  Precision: 0.757
  Recall   : 0.696
  F1-Score : 0.713

Random forest Performance:
  Accuracy : 0.911
  Precision: 0.709
  Recall   : 0.790
  F1-Score : 0.731

Logistic regression Performance:
  Accuracy : 0.889
  Precision: 0.572
  Recall   : 0.827
  F1-Score : 0.640


In [720]:
X_test['y_true'] = y_test.astype(int)
X_test['y_pred'] = y_pred.astype(int)
X_test['right'] = np.where(X_test['y_true'] == X_test['y_pred'], 'Yes', 'No')
X_test.head()

,season,year,athletes,avg_age,agv_height,avg_weight,sports,events,y_true,y_pred,right
1315,1,1968,39,25.0,174.0,66.0,4,22,0,0,Yes
3610,0,2002,4,24.0,169.0,64.0,2,4,0,0,Yes
521,1,1952,64,26.0,175.0,71.0,13,35,0,0,Yes
1469,1,1968,18,28.0,171.0,70.0,6,18,0,0,Yes
1756,1,2012,11,25.0,172.0,63.0,7,10,0,0,Yes


In [721]:
print(X_test['right'].value_counts(normalize=True))

right
Yes    0.946615
No     0.053385
Name: proportion, dtype: float64


In [722]:
mistakes = X_test[X_test['right'] == 'No']
mistakes

,season,year,athletes,avg_age,agv_height,avg_weight,sports,events,y_true,y_pred,right
102,1,1932,40,26.0,175.0,69.0,5,36,1,0,No
737,1,1948,127,26.0,176.0,71.0,13,83,0,1,No
75,1,1924,299,27.0,176.0,71.0,18,108,2,1,No
3585,0,2014,87,28.0,178.0,72.0,13,65,0,1,No
1094,1,1976,213,25.0,169.0,64.0,20,119,1,0,No
3052,0,1994,113,25.0,175.0,71.0,12,56,0,1,No
415,1,1996,164,26.0,179.0,77.0,15,84,1,0,No
3042,0,1998,113,27.0,175.0,73.0,13,54,1,0,No
882,1,1948,152,29.0,176.0,71.0,17,77,0,1,No
99,1,1920,63,32.0,176.0,72.0,9,51,1,0,No
